# TL9 — Aprendizaje no supervisado

## Índice

1. [Introducción](#1-introducción)
2. [Algoritmo EM para mixturas de Gaussianas](#2-algoritmo-em-para-mixturas-de-gaussianas)
3. [Algoritmo C-medias](#3-algoritmo-c-medias)
4. [Ejercicio con el EM para mixturas de Gaussianas](#4-ejercicio-con-el-em-para-mixturas-de-gaussianas)
5. [Ejercicio con el algoritmo C-medias](#5-ejercicio-con-el-algoritmo-c-medias)

> **Cita:** Juan Ciscar, Alfons. *"Material docente de la asignatura Aprendizaje Automático (APA)"*. Grado de Inteligencia Artificial (GIAR), Escuela Técnica Superior de Ingeniería Informática (ETSINF), Dep. de Sistemas Informáticos y Computación (DSIC), Universitat Politècnica de València (UPV), primer semestre del curso 2026/27.

---
## 0. Código auxiliar (Anexo: código T13)

En el PDF este código está en el **anexo** y los ejemplos lo cargan con `import import_ipynb; from T13 import *`. Aquí se incluye al principio para que el notebook funcione sin el archivo `T13.ipynb`: **ejecuta estas celdas antes que el resto**.

### 0.1 Librerías

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multinomial, multivariate_normal

np.random.seed(seed=17)

### 0.2 `pgmm`: densidad de un GMM evaluada en un *batch*

In [ ]:
def pgmm(X, pi, m, S):
    N = X.shape[0]
    Xp = np.zeros(N)
    C = pi.shape[0]
    for c in range(C):
        Xp += pi[c] * multivariate_normal(m[c], S[c]).pdf(X)
    return Xp

### 0.3 `genXy`: síntesis de datos a partir de $\boldsymbol{\pi}$, $\{\boldsymbol{\mu}_c\}$ y $\{\boldsymbol{\Sigma}_c\}$

In [ ]:
def genXy(N, pi, m, S):
    C, D = m.shape
    Nc = multinomial(1, pi).rvs(N-C).sum(0) + 1
    hpi = Nc / N
    X = []
    y = []
    for c in range(C):
        Xc = multivariate_normal(m[c], S[c]).rvs(Nc[c]).reshape(-1, D)
        X.append(Xc)
        yc = np.repeat(c, Nc[c]).reshape(-1, 1)
        y.append(yc)
    return np.vstack(X), np.vstack(y)

### 0.4 `GMM1d`: representación de un GMM en 1d

In [ ]:
def GMM1d(pi, m, S, ax):
    plt.grid()
    X = np.linspace(-2, 4, 200).reshape(-1, 1)
    C = pi.shape[0]
    for c in range(C):
        Xp = pgmm(X, np.array([pi[c]]), np.array([m[c]]), np.array([S[c]]))
        plt.plot(X, Xp, c="gray", linestyle="--")
    plt.plot(X, pgmm(X, pi, m, S))

### 0.5 `GMM2d`: representación de un GMM en 2d

In [ ]:
def GMM2d(pi, m, S, ax, x1lim=(-6, 6), x2lim=(-3.5, 3.5), X=None, y=None):
    C = pi.shape[0]
    x1, x2 = np.meshgrid(np.linspace(*x1lim, 200), np.linspace(*x2lim, 200))
    x12 = np.c_[np.ravel(x1), np.ravel(x2)]
    x12p = pgmm(x12, pi, m, S)
    ax.set_aspect('equal')
    ax.contour(x1, x2, x12p.reshape(x1.shape), 16, colors='orange', linestyles='solid')
    cp = ax.contourf(x1, x2, x12p.reshape(x1.shape), 16, cmap='Blues_r')
    plt.colorbar(cp, ax=ax, shrink=0.8)
    if X is not None:
        N = X.shape[0]
        s = 2 ** (7 - 2*np.log10(N))
        mask = (X[:, 0] >= x1lim[0]) & (X[:, 0] <= x1lim[1]) & (X[:, 1] >= x2lim[0]) & (X[:, 1] <= x2lim[1])
        ax.scatter(*X[mask, :].T, cmap='Set3', c=y[mask], s=s)

### 0.6 `PasoE`: cálculo de responsabilidades

In [ ]:
def PasoE(X, pi, m, S):
    N, D = X.shape
    C = pi.shape[0]
    R = np.zeros((N, C))
    for c in range(C):
        R[:, c] = pi[c] * multivariate_normal(m[c], S[c]).pdf(X)
    R /= R.sum(axis=1)[:, np.newaxis]
    return R

### 0.7 `PasoM`: actualización de parámetros

In [ ]:
def PasoM(X, R, Lambda=.0, diag=False):
    N, D = X.shape
    C = R.shape[1]
    Rc = R.sum(axis=0)
    pi = Rc / N
    m = R.T @ X / Rc[:, np.newaxis]
    S = np.zeros((C, D, D))
    G = np.zeros((D, D))
    for c in range(C):
        for n in range(N):
            xn = (X[n] - m[c]).reshape(-1, 1)
            S[c] += R[n, c] * xn @ xn.T
        S[c] /= Rc[c]
        if diag:
            S[c] = S[c] * np.eye(D)
        G += pi[c] * S[c]
    for c in range(C):
        S[c] = (1-Lambda) * S[c] + Lambda * G
    return pi, m, S

### 0.8 `hpimS`: MLE de $\boldsymbol{\pi}$, $\{\boldsymbol{\mu}_c\}$ y $\{\boldsymbol{\Sigma}_c\}$ con regularización

In [ ]:
def hpimS(X, C=2, Lambda=.0, diag=False, seed=23, max_iter=100):
    N, D = X.shape
    pi = np.ones(C) / C
    rng = np.random.default_rng(seed=seed)
    m = rng.permutation(X, axis=0)[:C]
    S = np.zeros((C, D, D))
    G = np.cov(X, rowvar=False, bias=True)
    for c in range(C):
        S[c] = G / C
    for i in range(max_iter):
        R = PasoE(X, pi, m, S)
        pi, m, S = PasoM(X, R, Lambda, diag)
    return pi, m, S

### 0.9 `GMM2dX`: `GMM2d` con aprendizaje por MLE previo

In [ ]:
def GMM2dX(pi, m, S, ax, x1lim=(-6, 6), x2lim=(-3.5, 3.5), N=100, Lambda=.0, diag=False, seed=23, max_iter=100):
    C = pi.shape[0]
    X, y = genXy(N, pi, m, S)
    hpi, hm, hS = hpimS(X, C=C, Lambda=Lambda, diag=diag, seed=seed, max_iter=max_iter)
    GMM2d(hpi, hm, hS, ax, x1lim=x1lim, x2lim=x2lim, X=X, y=y)

---
## 1. Introducción

**Caso no supervisado:** tenemos un dataset de "entradas sin salidas" $\mathcal{D} = \{\boldsymbol{x}_1, \dots, \boldsymbol{x}_N\}$.

**Algoritmo EM para mixturas de Gaussianas:** técnica estándar para ajustar $p(\boldsymbol{x} \mid \boldsymbol{\theta}) = \sum_c \pi_c\, \mathcal{N}(\boldsymbol{x} \mid \boldsymbol{\mu}_c, \boldsymbol{\Sigma}_c)$.

**Algoritmo C-medias:** técnica clásica de *clustering* que puede verse como un caso particular del EM.

**Evaluación con conteo de pares:** sean $U = \{U_1, \dots, U_R\}$ y $V = \{V_1, \dots, V_C\}$ particiones de $\mathcal{D}$:

- $N_{00}$: número de pares en diferentes clústers de $U$ y en diferentes clústers de $V$
- $N_{01}$: número de pares en el mismo clúster de $U$ y en diferentes clústers de $V$
- $N_{10}$: número de pares en diferentes clústers de $U$ y en el mismo clúster de $V$
- $N_{11}$: número de pares en el mismo clúster de $U$ y en el mismo clúster de $V$

**Rand index (RI):** número de pares tratados de la misma manera respecto al total de pares

$$
\operatorname{RI}(U, V) = \frac{N_{00} + N_{11}}{\binom{N}{2}} \qquad 0 \leq \operatorname{RI} \leq 1
$$

**Adjusted rand index (ARI):** RI con cero ajustado a su valor esperado con particiones aleatorias

$$
\operatorname{ARI}(U, V) = \frac{2(N_{00} N_{11} - N_{01} N_{10})}{(N_{00} + N_{01})(N_{01} + N_{11}) + (N_{00} + N_{10})(N_{10} + N_{11})} \qquad \operatorname{ARI} \leq 1
$$

**Objetivo:** aplicar EM y C-medias a tareas de clasificación y evaluar resultados con *clustering* (etiquetado) de referencia.

---
## 2. Algoritmo EM para mixturas de Gaussianas

**EM para GMMs en scikit-learn:** `GaussianMixture`

| Parámetro | Descripción |
|-----------|-------------|
| `n_components=1` | número de componentes |
| `covariance_type='full'` | otras opciones son `'tied'`, `'diag'` y `'spherical'` |
| `tol=0.001` | umbral de convergencia |
| `reg_covar=1e-06` | regularizador (aditivo) de varianzas |
| `max_iter=100` | número máximo de iteraciones EM |
| `n_init=1` | número de inicializaciones a realizar (se guardan los mejores resultados) |
| `init_params='kmeans'` | otras opciones son `'k-means++'`, `'random'` y `'random_from_data'` |

### 2.1 Ejemplo 1d

$C = 2$, $\pi_1 = \pi_2 = 0.5$, $\mu_1 = 0$, $\mu_2 = 2$, $\sigma_1^2 = \sigma_2^2 = 0.64$

Salida esperada:

```text
[0.5 0.5] [[0.] [2.]] [[[0.64]] [[0.64]]]
[0.6016 0.3984] [[1.2363] [2.2307]] [[[0.0677]] [[0.0267]]]
```

In [ ]:
np.set_printoptions(precision=4)

pi = .5*np.ones(2)
m = np.array([[0.], [2.]])
S = np.array([[[.64]], [[.64]]])
print(pi, str(m).replace('\n', ''), str(S).replace('\n', ''))

N = 5
X, y = genXy(N, pi, m, S)
y = y.reshape(-1)

from sklearn.mixture import GaussianMixture

gm = GaussianMixture(n_components=2, random_state=23).fit(X, y)
print(gm.weights_, str(gm.means_).replace('\n', ''), str(gm.covariances_).replace('\n', ''))

Evaluación con RI y ARI:

Salida esperada: `[0 0 1 1 1] [0 0 0 1 1] RI = 60.0% ARI = 16.7%`

In [ ]:
from sklearn.metrics import rand_score, adjusted_rand_score

y_pred = gm.predict(X)
RI = rand_score(y, y_pred)
ARI = adjusted_rand_score(y, y_pred)
print(f"{y} {y_pred} RI = {RI:.1%} ARI = {ARI:.1%}")

### 2.2 Ejemplo 2d

$C = 2$, $\pi_1 = \pi_2 = 0.5$, $\boldsymbol{\mu}_1 = (-1, 0)^t$, $\boldsymbol{\mu}_2 = (1, 0)^t$, $\boldsymbol{\Sigma}_1 = \boldsymbol{\Sigma}_2 = \sigma^2 \mathbf{I}_2$

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 3.5))

pi = .5*np.ones(2)
m = np.array([[-1, 0], [1, 0]])
N = 100
S = .2 * np.array([[[1, 0], [0, 1]], [[1, 0], [0, 1]]])
x1lim = (-2, 2)
x2lim = (-1, 1)

C = pi.shape[0]
X, y = genXy(N, pi, m, S)
GMM2d(pi, m, S, ax, x1lim=x1lim, x2lim=x2lim, X=X, y=y)
plt.show()

Búsqueda del número de componentes con `GridSearchCV` y ARI como métrica:

Salida esperada:

```text
ARI: 96.0% con {'covariance_type': 'full', 'max_iter': 20, 'n_components': 2}
Tiempo (hh:mm:ss): 00:00:00
```

In [ ]:
import time
import warnings
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import ShuffleSplit, GridSearchCV

warnings.filterwarnings('ignore')
start = time.time()

gm = GaussianMixture(random_state=23)
G = {'n_components': [2, 3, 4], 'covariance_type': ['full'], 'max_iter': [20]}
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23)
GS = GridSearchCV(gm, G, scoring='adjusted_rand_score', n_jobs=-1, cv=splitter, pre_dispatch=8)
y = y.reshape(-1)
ARI = GS.fit(X, y).score(X, y)

print(f"ARI: {ARI:.1%} con {GS.best_params_}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

### 2.3 MNIST: con los dígitos 6 y 9

Lectura, selección de las dos clases y reducción de la dimensión ($K = 400$).

Salida esperada: `(11867, 400) (11867,) (1967, 400) (1967,)`

In [ ]:
import numpy as np
import datasets
from sklearn.decomposition import PCA

sel = [6, 9]
ds = datasets.load_dataset("ylecun/mnist").with_format("numpy")
X_train = ds['train'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_train = ds['train'][:]['label'].astype(np.uint8)
mask = np.isin(y_train, sel)
X_train = X_train[mask, :]
y_train = y_train[mask]
X_test = ds['test'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_test = ds['test'][:]['label'].astype(np.uint8)
mask = np.isin(y_test, sel)
X_test = X_test[mask, :]
y_test = y_test[mask]

K = 400
pca = PCA(n_components=K)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

EM con búsqueda del número de componentes:

Salida esperada:

```text
ARI: 98.0% con {'covariance_type': 'full', 'max_iter': 100, 'n_components': 2}
Tiempo (hh:mm:ss): 00:00:21
```

In [ ]:
import time
import warnings
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import ShuffleSplit, GridSearchCV

warnings.filterwarnings('ignore')
start = time.time()

gm = GaussianMixture(random_state=23)
G = {'n_components': [2, 3, 4, 5, 6], 'covariance_type': ['full'], 'max_iter': [100]}
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23)
GS = GridSearchCV(gm, G, scoring='adjusted_rand_score', n_jobs=-1, cv=splitter, pre_dispatch=8)
ARI = GS.fit(X_train, y_train).score(X_test, y_test)

print(f"ARI: {ARI:.1%} con {GS.best_params_}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

---
## 3. Algoritmo C-medias

**Algoritmo C-medias en scikit-learn:** `k_means(X, n_clusters, *)`

| Parámetro | Descripción |
|-----------|-------------|
| `init='k-means++'` | otra opción es `'random'` |
| `n_init='auto'` | número de inicializaciones a realizar (se guardan los mejores resultados) |
| `max_iter=300` | número máximo de iteraciones |
| `tol=1e-4` | umbral de convergencia (Frobenius de la diferencia de centroides respecto a la iteración anterior) |
| `algorithm='lloyd'` | otra opción es `'elkan'` (más eficiente en tiempo) |

### 3.1 Ejemplo 1d

$C = 2$, $\pi_1 = \pi_2 = 0.5$, $\mu_1 = 0$, $\mu_2 = 2$, $\sigma_1^2 = \sigma_2^2 = 0.64$

Salida esperada:

```text
[0.5 0.5] [[0.] [2.]] [[[0.64]] [[0.64]]]
[[1.2339] [2.2304]] [0 0 0 1 1]
```

In [ ]:
np.set_printoptions(precision=4)

pi = .5*np.ones(2)
m = np.array([[0.], [2.]])
S = np.array([[[.64]], [[.64]]])
print(pi, str(m).replace('\n', ''), str(S).replace('\n', ''))

N = 5
X, y = genXy(N, pi, m, S)
y = y.reshape(-1)

from sklearn.cluster import k_means

means, y_pred, _ = k_means(X, 2, random_state=23)
print(str(means).replace('\n', ''), y_pred)

### 3.2 Ejemplo 2d

$C = 2$, $\pi_1 = \pi_2 = 0.5$, $\boldsymbol{\mu}_1 = (-1, 0)^t$, $\boldsymbol{\mu}_2 = (1, 0)^t$, $\boldsymbol{\Sigma}_1 = \boldsymbol{\Sigma}_2 = \sigma^2 \mathbf{I}_2$

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 3.5))

pi = .5*np.ones(2)
m = np.array([[-1, 0], [1, 0]])
N = 100
S = .2 * np.array([[[1, 0], [0, 1]], [[1, 0], [0, 1]]])
x1lim = (-2, 2)
x2lim = (-1, 1)

C = pi.shape[0]
X, y = genXy(N, pi, m, S)
GMM2d(pi, m, S, ax, x1lim=x1lim, x2lim=x2lim, X=X, y=y)
plt.show()

Búsqueda del número de clústers con ARI como métrica:

Salida esperada:

```text
ARI: 96.0% con {'n_clusters': 2, 'max_iter': 300}
Tiempo (hh:mm:ss): 00:00:00
```

In [ ]:
import time
import warnings
from sklearn.cluster import k_means
from sklearn.metrics import adjusted_rand_score

warnings.filterwarnings('ignore')
start = time.time()

y = y.reshape(-1)
best_ARI = .0
best_params = None
for n_clusters, max_iter in zip([2, 3, 4], [300]):
    means, y_pred, _ = k_means(X, n_clusters, max_iter=max_iter, random_state=23)
    ARI = adjusted_rand_score(y, y_pred)
    if ARI > best_ARI:
        best_ARI = ARI
        best_params = {'n_clusters': n_clusters, 'max_iter': max_iter}

print(f"ARI: {best_ARI:.1%} con {best_params}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

### 3.3 MNIST: con los dígitos 6 y 9

Salida esperada: `(11867, 400) (11867,) (1967, 400) (1967,)`

In [ ]:
import numpy as np
import datasets
from sklearn.decomposition import PCA

sel = [6, 9]
ds = datasets.load_dataset("ylecun/mnist").with_format("numpy")
X_train = ds['train'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_train = ds['train'][:]['label'].astype(np.uint8)
mask = np.isin(y_train, sel)
X_train = X_train[mask, :]
y_train = y_train[mask]
X_test = ds['test'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_test = ds['test'][:]['label'].astype(np.uint8)
mask = np.isin(y_test, sel)
X_test = X_test[mask, :]
y_test = y_test[mask]

K = 400
pca = PCA(n_components=K)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

C-medias en *train*; los datos de *test* se asignan al centroide más cercano (`NearestCentroid`).

Salida esperada:

```text
ARI: 93.6% con {'n_clusters': 2, 'max_iter': 300}
Tiempo (hh:mm:ss): 00:00:00
```

In [ ]:
import time
import warnings
from sklearn.cluster import k_means
from sklearn.neighbors import NearestCentroid
from sklearn.metrics import adjusted_rand_score

warnings.filterwarnings('ignore')
start = time.time()

y_train = y_train.reshape(-1)
best_ARI = .0
best_params = None
for n_clusters, max_iter in zip([2, 3, 4, 5, 6], [300]):
    means, y_train_pred, _ = k_means(X_train, n_clusters, max_iter=max_iter, random_state=23)
    y_test_pred = NearestCentroid().fit(means, np.arange(n_clusters)).predict(X_test)
    ARI = adjusted_rand_score(y_test, y_test_pred)
    if ARI > best_ARI:
        best_ARI = ARI
        best_params = {'n_clusters': n_clusters, 'max_iter': max_iter}

print(f"ARI: {best_ARI:.1%} con {best_params}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

---
## 4. Ejercicio con el EM para mixturas de Gaussianas

**Ejercicio:** aplica el algoritmo EM para mixturas de Gaussianas al par de clases (0, 1) de Fashion-MNIST y al par de clases (3, 9) de CIFAR-10, y trata de alcanzar (y mejorar) los resultados de referencia dados a continuación.

**ARI (%) en test:** 74.8 % en Fashion-MNIST y 48.3 % en CIFAR-10.

### 4.1 Fashion-MNIST: clases (0, 1), $K = 600$

#### 4.1.1 Lectura, selección de clases y reducción de la dimensión

Salida esperada: `(12000, 600) (12000,) (2000, 600) (2000,)`

In [ ]:
# Tu código aquí

#### 4.1.2 EM

Salida esperada:

```text
ARI: 74.8% con {'covariance_type': 'full', 'max_iter': 100, 'n_components': 2}
Tiempo (hh:mm:ss): 00:01:04
```

In [ ]:
# Tu código aquí

### 4.2 CIFAR-10: clases (3, 9), $K = 1000$

#### 4.2.1 Lectura, selección de clases y reducción de la dimensión

Salida esperada: `(10000, 1000) (10000,) (2000, 1000) (2000,)`

In [ ]:
# Tu código aquí

#### 4.2.2 EM

Salida esperada:

```text
ARI: 48.3% con {'covariance_type': 'full', 'max_iter': 100, 'n_components': 3}
Tiempo (hh:mm:ss): 00:01:17
```

In [ ]:
# Tu código aquí

---
## 5. Ejercicio con el algoritmo C-medias

**Ejercicio:** aplica el algoritmo C-medias al par de clases (0, 1) de Fashion-MNIST y al par de clases (3, 9) de CIFAR-10, y trata de alcanzar (y mejorar) los resultados de referencia dados a continuación.

**ARI (%) en test:** 43.7 % en Fashion-MNIST y 12.4 % en CIFAR-10.

### 5.1 Fashion-MNIST: clases (0, 1), $K = 600$

#### 5.1.1 Lectura, selección de clases y reducción de la dimensión

Salida esperada: `(12000, 600) (12000,) (2000, 600) (2000,)`

In [ ]:
# Tu código aquí

#### 5.1.2 C-medias

Salida esperada:

```text
ARI: 43.7% con {'n_clusters': 2, 'max_iter': 300}
Tiempo (hh:mm:ss): 00:00:00
```

In [ ]:
# Tu código aquí

### 5.2 CIFAR-10: clases (3, 9), $K = 1000$

#### 5.2.1 Lectura, selección de clases y reducción de la dimensión

Salida esperada: `(10000, 1000) (10000,) (2000, 1000) (2000,)`

In [ ]:
# Tu código aquí

#### 5.2.2 C-medias

Salida esperada:

```text
ARI: 12.4% con {'n_clusters': 2, 'max_iter': 300}
Tiempo (hh:mm:ss): 00:00:00
```

In [ ]:
# Tu código aquí